## Building a Q&A app with graph db

In [32]:
import os
from dotenv import load_dotenv
load_dotenv('/home/abhi/AI_Workspace/personal/Generative-AI-Engineer-Portfolio/.env')

#I am using local docker instance of neo4j, you can also use neo4j cloud instance by providing the connection details in .env file. Instead I am defining it here for local instance.
NEO4J_URI="bolt://localhost:7687"
NEO4J_USERNAME="neo4j"
NEO4J_PASSWORD="neoadmin"
NEO4J_DATABASE="my-test-db"


In [74]:
from langchain_neo4j import Neo4jGraph
graph = Neo4jGraph(url=NEO4J_URI, username=NEO4J_USERNAME, password=NEO4J_PASSWORD, database=NEO4J_DATABASE)
graph

In [78]:
movie_query = """
LOAD CSV WITH HEADERS FROM 'file:///movies_small.csv' AS row
MERGE (m:Movie {id: row.movieId})
SET m.released = date(row.released),
    m.title = row.title,
    m.imdbRating = toFloat(row.imdbRating)
FOREACH (director IN split(row.director, '|') |
    MERGE (p:Person {name: trim(director)})
    MERGE (p)-[:DIRECTED]->(m)
)
FOREACH (actor IN split(row.actors, '|') |
    MERGE (p:Person {name: trim(actor)})
    MERGE (p)-[:ACTED_IN]->(m)
)
FOREACH (genre IN split(row.genres, '|') |
    MERGE (g:Genre {name: trim(genre)})
    MERGE (g)-[:IN_GENRE]->(m)
)
"""
movie_query

"\nLOAD CSV WITH HEADERS FROM 'file:///movies_small.csv' AS row\nMERGE (m:Movie {id: row.movieId})\nSET m.released = date(row.released),\n    m.title = row.title,\n    m.imdbRating = toFloat(row.imdbRating)\nFOREACH (director IN split(row.director, '|') |\n    MERGE (p:Person {name: trim(director)})\n    MERGE (p)-[:DIRECTED]->(m)\n)\nFOREACH (actor IN split(row.actors, '|') |\n    MERGE (p:Person {name: trim(actor)})\n    MERGE (p)-[:ACTED_IN]->(m)\n)\nFOREACH (genre IN split(row.genres, '|') |\n    MERGE (g:Genre {name: trim(genre)})\n    MERGE (g)-[:IN_GENRE]->(m)\n)\n"

In [79]:
graph.query(movie_query)

[]

In [80]:
graph.refresh_schema()
print(graph.schema)

Node properties:
Movie {imdbRating: FLOAT, id: STRING, released: DATE, title: STRING}
Person {name: STRING}
Genre {name: STRING}
Relationship properties:

The relationships:
(:Person)-[:ACTED_IN]->(:Movie)
(:Person)-[:DIRECTED]->(:Movie)
(:Genre)-[:IN_GENRE]->(:Movie)


In [ ]:
groq_api_key = os.getenv('GROQ_API_KEY')
print(groq_api_key)

In [82]:
from langchain_groq import ChatGroq
chat_model = ChatGroq(api_key=groq_api_key, model="llama-3.1-8b-instant")
chat_model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x71bdf9f83dd0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x71bdf9f80950>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [83]:
from langchain_neo4j import GraphCypherQAChain
qa_chain = GraphCypherQAChain.from_llm(llm=chat_model, graph=graph, verbose=True, allow_dangerous_requests=True)
qa_chain

GraphCypherQAChain(verbose=True, graph=<langchain_neo4j.graphs.neo4j_graph.Neo4jGraph object at 0x71be3a8bd290>, cypher_generation_chain=PromptTemplate(input_variables=['examples', 'question', 'schema'], input_types={}, partial_variables={}, template='Task:Generate Cypher statement to query a graph database.\nInstructions:\nUse only the provided relationship types and properties in the schema.\nDo not use any other relationship types or properties that are not provided.\nSchema:\n{schema}\nNote: Do not include any explanations or apologies in your responses.\nDo not respond to any questions that might ask anything else than for you to construct a Cypher statement.\nDo not include any text except the generated Cypher statement.\n\nExamples (optional):\n{examples}\n\nThe question is:\n{question}')
| RunnableBinding(bound=ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'au

In [84]:
response = qa_chain.run("Which movies were released after 1995?")
print(response)



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (m:Movie) WHERE m.released > '1995-01-01' RETURN m
Full Context:
[]

> Finished chain.
I don't know the answer.


In [86]:
response = qa_chain.run("Who directed the movie Casino?")
print(response)



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person)-[:DIRECTED]->(m:Movie {title: "Casino"}) RETURN p
Full Context:
[{'p': {'name': 'Martin Scorsese'}}]

> Finished chain.
Martin Scorsese directed the movie Casino.


In [87]:
#Exclude nodes of type Genre from the graph that the LLM can see.
new_chain = GraphCypherQAChain.from_llm(llm=chat_model, graph=graph, exclude_types=["Genre"], verbose=True, allow_dangerous_requests=True)

new_chain.graph_schema

'Node properties:\nMovie {imdbRating: FLOAT, id: STRING, released: DATE, title: STRING}\nPerson {name: STRING}\nRelationship properties:\n\nThe relationships:\n(:Person)-[:ACTED_IN]->(:Movie)\n(:Person)-[:DIRECTED]->(:Movie)'

In [97]:
examples = [
    {
        "question": "How many actors are there?",
        "query": "MATCH (a:Person)-[:ACTED_IN]->(:Movie) RETURN count(DISTINCT a)",
    },
    {
        "question": "How many movies are there?",
        "query": "MATCH (m:Movie) RETURN count(m)",
    },
    {
        "question": "What genres exist?",
        "query": "MATCH (g:Genre) RETURN DISTINCT g.name",
    },
    {
        "question": "Who are the directors?",
        "query": "MATCH (p:Person)-[:DIRECTED]->(:Movie) RETURN DISTINCT p.name",
    },
]

In [103]:
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate

example_prompt = PromptTemplate.from_template(
    "User Input: {question}\nCypher Query: {query}"
)

prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    prefix="You are a Neo4j expert. Generate a syntactically correct Cypher query only.\nDo not include any explanations, apologies, or text.\nOnly output the Cypher query.",
    suffix="User Input: {question}\nCypher Query:",
    input_variables=["question"],
)

In [104]:
prompt

FewShotPromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, examples=[{'question': 'How many actors are there?', 'query': 'MATCH (a:Person)-[:ACTED_IN]->(:Movie) RETURN count(DISTINCT a)'}, {'question': 'How many movies are there?', 'query': 'MATCH (m:Movie) RETURN count(m)'}, {'question': 'What genres exist?', 'query': 'MATCH (g:Genre) RETURN DISTINCT g.name'}, {'question': 'Who are the directors?', 'query': 'MATCH (p:Person)-[:DIRECTED]->(:Movie) RETURN DISTINCT p.name'}], example_prompt=PromptTemplate(input_variables=['query', 'question'], input_types={}, partial_variables={}, template='User Input: {question}\nCypher Query: {query}'), suffix='User Input: {question}\nCypher Query:', prefix='You are a Neo4j expert. Generate a syntactically correct Cypher query only.\nDo not include any explanations, apologies, or text.\nOnly output the Cypher query.')

In [105]:
print(prompt.format(question="How many artists are there?", schema="foo"))

You are a Neo4j expert. Generate a syntactically correct Cypher query only.
Do not include any explanations, apologies, or text.
Only output the Cypher query.

User Input: How many actors are there?
Cypher Query: MATCH (a:Person)-[:ACTED_IN]->(:Movie) RETURN count(DISTINCT a)

User Input: How many movies are there?
Cypher Query: MATCH (m:Movie) RETURN count(m)

User Input: What genres exist?
Cypher Query: MATCH (g:Genre) RETURN DISTINCT g.name

User Input: Who are the directors?
Cypher Query: MATCH (p:Person)-[:DIRECTED]->(:Movie) RETURN DISTINCT p.name

User Input: How many artists are there?
Cypher Query:


In [109]:
from langchain_core.prompts import PromptTemplate as PT

# Custom QA prompt to format answers properly
qa_prompt = PT.from_template(
    """You are a helpful assistant that answers questions based on graph database query results.
Given the query results, provide a clear and concise answer to the question.
If the result is empty, say "I don't have that information in the database."

Question: {question}
Graph Query Result: {context}

Answer:"""
)

cypher_prompt_chain = GraphCypherQAChain.from_llm(
    llm=chat_model, 
    graph=graph, 
    cypher_prompt=prompt,
    qa_prompt=qa_prompt,
    verbose=True, 
    allow_dangerous_requests=True
)

In [110]:
cypher_prompt_chain.invoke("How many actors are there?")



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (a:Person)-[:ACTED_IN]->(:Movie) RETURN count(DISTINCT a)
Full Context:
[{'count(DISTINCT a)': 967}]

> Finished chain.


{'query': 'How many actors are there?', 'result': 'There are 967 actors.'}

In [111]:
cypher_prompt_chain.invoke("How many movies has Tom Hanks acted in?")



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (a:Person {name: "Tom Hanks"})-[:ACTED_IN]->(m:Movie) RETURN count(m)
Full Context:
[{'count(m)': 2}]

> Finished chain.


{'query': 'How many movies has Tom Hanks acted in?',
 'result': 'Tom Hanks has acted in 2 movies.'}

In [112]:
cypher_prompt_chain.invoke("Which director has directed the most movies?")



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person)-[:DIRECTED]->(m:Movie) RETURN p.name, COUNT(m) AS num_movies ORDER BY num_movies DESC LIMIT 1
Full Context:
[{'p.name': 'Robert Rodriguez', 'num_movies': 3}]

> Finished chain.


{'query': 'Which director has directed the most movies?',
 'result': 'Robert Rodriguez has directed the most movies with 3 films.'}

In [113]:
cypher_prompt_chain.invoke("How many directors are there?")



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person)-[:DIRECTED]->(:Movie) RETURN count(DISTINCT p)
Full Context:
[{'count(DISTINCT p)': 290}]

> Finished chain.


{'query': 'How many directors are there?',
 'result': 'There are 290 directors.'}

In [114]:
cypher_prompt_chain.invoke("Who are the directors with whom Tom Hanks has worked in movies?")



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (t:Person {name: "Tom Hanks"})-[:ACTED_IN]->(m:Movie)<-[:DIRECTED]-(d:Person) RETURN DISTINCT d.name
Full Context:
[{'d.name': 'John Lasseter'}, {'d.name': 'Ron Howard'}]

> Finished chain.


{'query': 'Who are the directors with whom Tom Hanks has worked in movies?',
 'result': 'Tom Hanks has worked with directors John Lasseter and Ron Howard in movies.'}